In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data=pd.read_excel('marketing_campaign1.xlsx')

In [ ]:
data

In [ ]:
data.shape

In [ ]:
data.size

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
data.shape

In [ ]:
data.size

# **Explorartory Data Analysis**

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().any()

In [ ]:
# fix missing value in column Income / using group by Education and Marital Status / avg
missing=data.groupby(["Education","Marital_Status"])["Income"].transform("mean").round(0)
data["Income"].fillna(missing, inplace= True)

In [ ]:
# Label Encoding
from sklearn.preprocessing import LabelEncoder
LE = LabelEncoder()
data["Education"] = LE.fit_transform(data["Education"])
data["Marital_Status"] = LE.fit_transform(data["Marital_Status"])

In [ ]:
present_year = 2024
data['Age'] = present_year-data['Year_Birth']
data['Age']

In [ ]:
data.columns

In [ ]:
columns_to_drop = ['ID', 'Year_Birth','Dt_Customer', 'Z_CostContact', 'Z_Revenue', 'Response']
data = data.drop(columns=columns_to_drop)
data

In [ ]:
data.shape

In [ ]:
data[['Age', 'Income', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts','MntSweetProducts', 'MntGoldProds',
     'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth']]

In [ ]:
# fix missing value in column Income / using group by Education and Marital Status / avg
missing=data.groupby(["Education","Marital_Status"])["Income"].transform("mean").round(0)
data["Income"].fillna(missing, inplace= True)

In [ ]:
# Define the columns to remove outliers
columns_to_check = ['Age', 'Income', 'MntWines', 'MntFruits', 'MntMeatProducts','MntFishProducts', 'MntSweetProducts', 'MntGoldProds',
                  'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumWebVisitsMonth']

In [ ]:
# Function to remove outliers using z-score
from scipy import stats
def remove_outliers(data, columns):
    threshold = 3
    z_scores = stats.zscore(data[columns])
    # Filter rows where any z-score exceeds the threshold
    return data[(z_scores < threshold).all(axis=1)]

# Remove outliers using z-score
data1= remove_outliers(data, columns_to_check)

# Compare the size of the original dataset with datasets after removing outliers
print("Original dataset size:", data1.shape)
print("Dataset size after removing outliers (z_score)):", df_no_outliers.shape)

In [ ]:
data1.info()

In [ ]:
data1.drop_duplicates(inplace=True)

In [ ]:
data1.shape

# **Visualizations**

In [ ]:
# correlation matrix
import matplotlib.pyplot as plt
import seaborn as sn
corr = data1.corr()
plt.figure(figsize=[20, 10])
sn.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5,linecolor='black')

plt.title('Correlation Matrix', fontsize=16)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.show()

In [ ]:
data1['Income'].plot(kind="hist")
plt.show()

In [ ]:
data1['Marital_Status'].value_counts().plot(kind="bar")
plt.show()

In [ ]:
data1['Education'].value_counts().plot(kind="bar")
plt.show()

In [ ]:
# Label Encoding
from sklearn.preprocessing import LabelEncoder
LE = LabelEncoder()
data1["Education"] = LE.fit_transform(data1["Education"])
data1["Marital_Status"] = LE.fit_transform(data1["Marital_Status"])

In [ ]:
data1

In [ ]:
data1.boxplot(column="Education")
plt.show()

# **Standardization Data**

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
data1=pd.DataFrame(scaler.fit_transform(data1.iloc[:,0:]),columns=data.columns)

In [ ]:
data1

# **Normalization data**

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
data_normalized = pd.DataFrame(scaler.fit_transform(data1.iloc[:, 0:]), columns=data.columns)

In [ ]:
data_normalized

# **Clustering**
# KMeans

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.cluster import KMeans
fig=plt.figure(figsize=(10,6))
wcss=[]
for i in range (1,11):
  clf=KMeans(n_clusters=i)
  clf.fit(data_normalized )
  wcss.append(clf.inertia_)
plt.scatter(range(1,11),wcss)
plt.plot(range(1,11),wcss)
plt.title('The Elbow Method')
plt.xlabel('number if clusters')
plt.ylabel('wcss')
plt.show()

In [ ]:
clf=KMeans(n_clusters=5)
y_KMeans=clf.fit_predict(data_normalized)

In [ ]:
y_KMeans=clf.labels_
y_KMeans

In [ ]:
clf.cluster_centers_

In [ ]:
clf.inertia_

In [ ]:
data1['Cluster']=y_KMeans
data1

In [ ]:
data1['Cluster']

In [ ]:
import seaborn as sn
fig = plt.figure(figsize=(10,6))
sn.countplot(x=data1["Cluster"],color='maroon')
plt.title("Distribution Of The Clusters")
plt.show()

In [ ]:
a=data1[(data1.Cluster==0)]
a

In [ ]:
a=data1[(data1.Cluster==1)]
a

In [ ]:
a=data1[(data1.Cluster==2)]
a

In [ ]:
a=data1[(data1.Cluster==3)]
a

In [ ]:
a=data1[(data1.Cluster==4)]
a

In [ ]:
plt.style.use(['classic'])
print(plt.style.available)

In [ ]:
data1.plot(x='Income',y='Age',c=clf.labels_,kind='scatter',s=70,cmap=plt.cm.coolwarm)
plt.title('Clusters using kmeans')
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

# Calculate the silhouette score for the KMeans clustering
silhouette_avg = silhouette_score(data_normalized, y_KMeans)

# Print the silhouette score
print("Silhouette score", silhouette_avg)

# **Model Validation**

# **Splitting Data into Training and Testing Set**

In [ ]:
x=data1.iloc[:,0:19]
y=data1['Cluster']

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y, test_size=0.30)

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)


# **Model Building**

# **Decision Tree**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
DT_model=DecisionTreeClassifier()
DT_model.fit(x_train,y_train)

In [ ]:
# Testing the model
y_predDT=DT_model.predict(x_test)
y_predDT

In [ ]:
#Confusion Matrix and Accuracy score
from sklearn.metrics import confusion_matrix,accuracy_score
DT=confusion_matrix(y_test,y_predDT)
print('Confusion_Matrix:','\n',DT)


DT_model= DecisionTreeClassifier(random_state=42, min_samples_leaf=5)
DT_model.fit(x_train, y_train)
y_pred = DT_model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy Score:", accuracy)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,DT_model.predict(x_test)))

****

In [ ]:
from sklearn.ensemble import RandomForestClassifier
RF_model=RandomForestClassifier(n_estimators=100, criterion='entropy', random_state=0)
RF_model.fit(x_train,y_train)

In [ ]:
# Testing the model
y_predRF=RF_model.predict(x_test)
y_predRF

In [ ]:
# Confusion Matrix and Accuracy score
RF = confusion_matrix(y_test, y_predRF)
print('Confusion_Matrix:','\n', RF)
RF_model = RandomForestClassifier(n_estimators=100, criterion='entropy', random_state=42, min_samples_leaf=5)
RF_model.fit(x_train, y_train)
y_pred = RF_model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy Score:", accuracy)

In [ ]:
print(classification_report(y_test,RF_model.predict(x_test)))

# **Ada Boost**

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada_model = AdaBoostClassifier(n_estimators=100, learning_rate=0.1,random_state=42)
ada_model.fit(x_train, y_train)

In [ ]:
y_pred_ada = ada_model.predict(x_test)
y_pred_ada

In [ ]:
ada_cm = confusion_matrix(y_test, y_pred_ada)
print('Confusion_Matrix:','\n', ada_cm)
print("Accuracy Score:", accuracy)

In [ ]:
print(classification_report(y_test, ada_model.predict(x_test)))

# **XGBoost**

In [ ]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(x_train,y_train)

In [ ]:
# Testing the model
y_predxgb=xgb_model.predict(x_test)
y_predxgb

In [ ]:
#Confusion Matrix and Accuracy
from sklearn.metrics import confusion_matrix,accuracy_score
xgb=confusion_matrix(y_test,y_predxgb)
print('Confusion_Matrix:','\n',xgb_model)
d=accuracy_score(y_test,y_predxgb)
print('Accuracy_Score:', d)

In [ ]:
print(classification_report(y_test,xgb_model.predict(x_test)))

# **Naive Bayes Classifier**

In [ ]:
from sklearn.naive_bayes import GaussianNB
nb_model = GaussianNB()
nb_model.fit(x_train, y_train)
# Testing the model

In [ ]:
# Testing the model
y_prednb=nb_model.predict(x_test)
y_prednb

In [ ]:
#Confusion Matrix and Accuracy
nb=confusion_matrix(y_test,y_prednb)
print('Confusion_Matrix:','\n',nb)
d=accuracy_score(y_test,y_prednb)
print('Accuracy_Score:', d)

In [ ]:
print(classification_report(y_test,nb_model.predict(x_test)))

# **K-Nearest Neighbors**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
KNN_model = KNeighborsClassifier(n_neighbors=6)
KNN_model.fit(x_train, y_train)

In [ ]:
# Testing the model
y_predKNN=KNN_model.predict(x_test)
y_predKNN

In [ ]:
#Confusion Matrix and Accuracy
from sklearn.metrics import confusion_matrix,accuracy_score
KNN=confusion_matrix(y_test,y_predKNN)
print('Confusion_Matrix:','\n',KNN_model)
bc=accuracy_score(y_test,y_predKNN)
print('Accuracy_Score:', bc)

In [ ]:
print(classification_report(y_test,KNN_model.predict(x_test)))

# **Support Vector Machines(SVM)**

In [ ]:
from sklearn.svm import SVC
SVM_model=SVC()
SVM_model.fit(x_train,y_train)

In [ ]:
# Testing the model
y_predSVM=SVM_model.predict(x_test)
y_predSVM

In [ ]:
#Confusion Matrix and Accuracy
SVM=confusion_matrix(y_test,y_predSVM)
print(SVM)
accuracy_score(y_test,y_predSVM)

In [ ]:
print(classification_report(y_test,SVM_model.predict(x_test)))

In [ ]:
# save model using pickle and load and predict
from pickle import dump
from pickle import load
import pickle

# **Saving the trained model**

In [ ]:
filename='classifier.pkl.sav'
pickle.dump(xgb_model,open(filename,'wb'))

In [ ]:
# loading the saved model
loaded_model=pickle.load(open(filename,'rb'))

In [ ]:
result=loaded_model.score(x_test,y_test)
print(result)